In [ ]:
import os
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import CLIPProcessor, CLIPModel
from PIL import Image
from tqdm import tqdm
import numpy as np
import pandas as pd
import random
import gc
from sklearn.metrics import cohen_kappa_score

# ==========================================
# 0. CONFIGURATION & SEED
# ==========================================
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True

set_seed(42)

def device():
    if torch.cuda.is_available(): return "cuda"
    elif torch.xpu.is_available(): return "xpu"
    elif torch.backends.mps.is_available(): return "mps"
    return "cpu"

# ----------------- HYPERPARAMETER SETTINGS -----------------
MODEL_NAME = "openai/clip-vit-base-patch32"
IMAGE_FOLDER = "images"           # Folder tempat image files disimpan
BATCH_SIZE = 8                    
EPOCHS = 10                       
UNFREEZE_LAYERS = 3               
LR_CLIP = 5e-6                    
LR_HEADS = 1e-3                   
NUM_WORKERS = 0                   
MAX_CHUNKS = 7                   

DEVICE = device()
print(f"🔥 Using device: {DEVICE}")

TRAITS = [
    'organizational_structure(ground_truth)', 'coherence(ground_truth)', 
    'essay_length(ground_truth)', 'grammatical_accuracy(ground_truth)', 
    'grammatical_diversity(ground_truth)', 'lexical_accuracy(ground_truth)', 
    'lexical_diversity(ground_truth)', 'punctuation_accuracy(ground_truth)',
    'argument_clarity(ground_truth)', 'justifying_persuasiveness(ground_truth)' 
]

# ==========================================
# 1. METRICS
# ==========================================
def compute_qwk(preds, targets):
    preds_rounded = np.round(preds * 2).astype(int)
    targets_rounded = np.round(targets * 2).astype(int)
    try:
        qwk = cohen_kappa_score(targets_rounded, preds_rounded, weights="quadratic")
    except:
        qwk = 0.0
    return qwk

# ==========================================
# 2. CUSTOM DATASET DENGAN SLIDING WINDOW
# ==========================================
class MultimodalDatasetLokal(Dataset):
    def __init__(self, df, processor, image_root=IMAGE_FOLDER, max_chunks=MAX_CHUNKS):
        self.data = df.reset_index(drop=True)
        self.processor = processor
        self.image_root = image_root
        self.max_chunks = max_chunks

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        row = self.data.iloc[idx]
        
        text = f"Question: {row['Question']} Answer: {str(row['Essay'])}"
        
        image_idx = int(row['local_image_id'])
        image_name = f"{image_idx}.jpg" 
        image_path = os.path.join(self.image_root, image_name)

        try:
            if os.path.exists(image_path):
                image = Image.open(image_path).convert("RGB")
            else:
                image = Image.new('RGB', (224, 224), (255, 255, 255))
        except Exception:
            image = Image.new('RGB', (224, 224), (255, 255, 255))
            
        labels = [float(row[t]) for t in TRAITS]


        # Tokenize seluruh teks tanpa memotongnya
        tokens = self.processor.tokenizer(text, truncation=False, return_tensors=None, add_special_tokens=False)
        raw_input_ids = tokens['input_ids']
        
        # Standar CLIP: max 77 token (1 BOS + 75 text + 1 EOS)
        chunk_size = 75 
        chunks_ids = []
        chunks_attn = []
        chunk_flags = [] # Menyimpan flag: 1 jika chunk berisi teks asli, 0 jika hanya padding
        
        # Pecah teks menjadi beberapa chunk
        for i in range(0, len(raw_input_ids), chunk_size):
            chunk = raw_input_ids[i : i + chunk_size]
            chunk = [self.processor.tokenizer.bos_token_id] + chunk + [self.processor.tokenizer.eos_token_id]
            
            pad_len = 77 - len(chunk)
            mask = [1] * len(chunk) + [0] * pad_len
            
            # Tambahkan pad token
            # Pada CLIP, pad_token seringkali sama dengan eos_token
            pad_token = self.processor.tokenizer.pad_token_id if self.processor.tokenizer.pad_token_id is not None else self.processor.tokenizer.eos_token_id
            chunk = chunk + [pad_token] * pad_len
            
            chunks_ids.append(chunk)
            chunks_attn.append(mask)
            chunk_flags.append(1.0)
            
        # Jika teks kosong sama sekali
        if len(chunks_ids) == 0:
            pad_token = self.processor.tokenizer.pad_token_id if self.processor.tokenizer.pad_token_id is not None else self.processor.tokenizer.eos_token_id
            chunk = [self.processor.tokenizer.bos_token_id, self.processor.tokenizer.eos_token_id] + [pad_token] * 75
            mask = [1, 1] + [0] * 75
            chunks_ids.append(chunk)
            chunks_attn.append(mask)
            chunk_flags.append(1.0)
            
        # Potong jika melebihi batas MAX_CHUNKS
        chunks_ids = chunks_ids[:self.max_chunks]
        chunks_attn = chunks_attn[:self.max_chunks]
        chunk_flags = chunk_flags[:self.max_chunks]
        
        # Pad sisa array jika jumlah chunk kurang dari MAX_CHUNKS
        while len(chunks_ids) < self.max_chunks:
            pad_token = self.processor.tokenizer.pad_token_id if self.processor.tokenizer.pad_token_id is not None else self.processor.tokenizer.eos_token_id
            chunk = [self.processor.tokenizer.bos_token_id, self.processor.tokenizer.eos_token_id] + [pad_token] * 75
            mask = [1, 1] + [0] * 75
            chunks_ids.append(chunk)
            chunks_attn.append(mask)
            chunk_flags.append(0.0) # Flag 0 karena ini purely padding chunk
            
        # Proses gambar saja dengan CLIPProcessor
        image_encoding = self.processor(images=image, return_tensors="pt")
        
        return {
            "input_ids": torch.tensor(chunks_ids, dtype=torch.long),
            "attention_mask": torch.tensor(chunks_attn, dtype=torch.long),
            "chunk_flags": torch.tensor(chunk_flags, dtype=torch.float),
            "pixel_values": image_encoding["pixel_values"].squeeze(0),
            "labels": torch.tensor(labels, dtype=torch.float)
        }

# ==========================================
# 3. MODEL ARCHITECTURE DENGAN CHUNK AGGREGATION
# ==========================================
class MultimodalCLIP_AES(nn.Module):
    def __init__(self, model_name=MODEL_NAME, unfreeze_layers=UNFREEZE_LAYERS):
        super().__init__()
        self.clip = CLIPModel.from_pretrained(model_name)
        
        for param in self.clip.parameters():
            param.requires_grad = False
            
        for layer in self.clip.vision_model.encoder.layers[-unfreeze_layers:]:
            for param in layer.parameters(): param.requires_grad = True
                
        for layer in self.clip.text_model.encoder.layers[-unfreeze_layers:]:
            for param in layer.parameters(): param.requires_grad = True
                
        for param in self.clip.visual_projection.parameters(): param.requires_grad = True
        for param in self.clip.text_projection.parameters(): param.requires_grad = True

        self.heads = nn.ModuleList([
            nn.Sequential(
                nn.Linear(1024, 256),
                nn.ReLU(),
                nn.Dropout(0.2),
                nn.Linear(256, 1)
            ) for _ in range(len(TRAITS))
        ])

    def forward(self, input_ids, attention_mask, chunk_flags, pixel_values):
  
        bsz, max_chunks, seq_len = input_ids.shape
        
        # Flatten menjadi (Batch * Max_Chunks, 77) agar bisa diproses CLIP sekaligus
        flat_input_ids = input_ids.view(bsz * max_chunks, seq_len)
        flat_attention_mask = attention_mask.view(bsz * max_chunks, seq_len)
        
        # Ekstrak fitur teks
        text_outputs = self.clip.text_model(input_ids=flat_input_ids, attention_mask=flat_attention_mask)
        text_projected = self.clip.text_projection(text_outputs.pooler_output)
        
        # Kembalikan ke bentuk (Batch, Max_Chunks, Embedding_Dim)
        text_projected = text_projected.view(bsz, max_chunks, -1)
        
        # Lakukan Mean Pooling hanya pada chunk yang valid (flag=1)
        chunk_flags = chunk_flags.unsqueeze(-1) # Shape: (Batch, Max_Chunks, 1)
        sum_text_embeds = (text_projected * chunk_flags).sum(dim=1)
        valid_chunk_count = chunk_flags.sum(dim=1).clamp(min=1e-9)
        
        # Vektor teks akhir (Rata-rata dari seluruh chunk)
        text_embeds = sum_text_embeds / valid_chunk_count
        text_embeds = F.normalize(text_embeds, p=2, dim=-1)
        
        # Ekstrak fitur visual
        vision_outputs = self.clip.vision_model(pixel_values=pixel_values)
        vision_projected = self.clip.visual_projection(vision_outputs.pooler_output)
        vision_embeds = F.normalize(vision_projected, p=2, dim=-1)
        
        # Gabungkan vektor Teks + Visual
        fused_features = torch.cat((text_embeds, vision_embeds), dim=1) 
        preds = [head(fused_features) for head in self.heads]
        return torch.cat(preds, dim=1)

# ==========================================
# 4. MAIN LOOP (TRAIN & PREDICT)
# ==========================================
def main():
    print("Loading Processor & Preparing Data...")
    processor = CLIPProcessor.from_pretrained(MODEL_NAME)
    subset_drop = ['graph', 'Essay'] + TRAITS
    
    # Load predefined train/test split yang sudah disiapkan
    print("📂 Loading predefined data split dari train_df.csv dan test_df.csv...")
    df_train = pd.read_csv('train_df.csv', encoding='latin1')
    df_test = pd.read_csv('test_df.csv', encoding='latin1')
    
    df_train = df_train.dropna(subset=subset_drop).reset_index(drop=True)
    df_test = df_test.dropna(subset=subset_drop).reset_index(drop=True)
    
    # Assign local_image_id untuk mapping gambar
    df_train['local_image_id'] = range(len(df_train))
    df_test['local_image_id'] = range(len(df_train), len(df_train) + len(df_test))
    
    print(f"✅ Train set: {len(df_train)} samples | Test set: {len(df_test)} samples\n")
    
    train_dataset = MultimodalDatasetLokal(df_train, processor, image_root=IMAGE_FOLDER)
    test_dataset = MultimodalDatasetLokal(df_test, processor, image_root=IMAGE_FOLDER)
    
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS)
    test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS)
    
    print(f"Dataset siap: {len(train_dataset)} Train | {len(test_dataset)} Test\n")

    print("Initializing Model & Optimizer...")
    model = MultimodalCLIP_AES().to(DEVICE)
    
    clip_unfreezed_params = []
    custom_heads_params = []
    for name, param in model.named_parameters():
        if not param.requires_grad: continue
        if "clip" in name: clip_unfreezed_params.append(param)
        else: custom_heads_params.append(param)

    optimizer = optim.AdamW([
        {"params": clip_unfreezed_params, "lr": LR_CLIP},
        {"params": custom_heads_params, "lr": LR_HEADS}
    ], weight_decay=0.01)

    criterion = nn.MSELoss()

    # --- TRAINING PHASE ---
    print("\n" + "="*50)
    print("🚀 MULAI TRAINING")
    print("="*50)
    for epoch in range(EPOCHS):
        gc.collect()
        if DEVICE == "xpu": torch.xpu.empty_cache()
        elif DEVICE == "cuda": torch.cuda.empty_cache()
        
        model.train()
        total_train_loss = 0
        loop = tqdm(train_loader, leave=True, desc=f"Epoch [{epoch+1}/{EPOCHS}] Train")
        
        for batch in loop:
            input_ids = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            chunk_flags = batch["chunk_flags"].to(DEVICE)
            pixel_values = batch["pixel_values"].to(DEVICE)
            labels = batch["labels"].to(DEVICE)
            
            optimizer.zero_grad()
            preds = model(input_ids, attention_mask, chunk_flags, pixel_values)
            loss = criterion(preds, labels)
            loss.backward()
            optimizer.step()
            
            total_train_loss += loss.item()
            loop.set_postfix(loss=loss.item())
            
        avg_train_loss = total_train_loss / len(train_loader)
        print(f"Epoch {epoch+1} Selesai | Avg Loss: {avg_train_loss:.4f}\n")

    # --- TESTING & PREDICTION PHASE ---
    print("="*50)
    print("🔍 MULAI TESTING PADA DATA UNSEEN")
    print("="*50)
    model.eval()
    
    all_preds = []
    all_targets = []
    
    with torch.no_grad():
        for batch in tqdm(test_loader, leave=True, desc="Testing"):
            input_ids = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            chunk_flags = batch["chunk_flags"].to(DEVICE)
            pixel_values = batch["pixel_values"].to(DEVICE)
            labels = batch["labels"].cpu().numpy()
            
            preds = model(input_ids, attention_mask, chunk_flags, pixel_values).cpu().numpy()
            
            all_preds.append(preds)
            all_targets.append(labels)

    all_preds = np.vstack(all_preds)
    all_targets = np.vstack(all_targets)
    
    # Kalkulasi QWK Score
    print("\n" + "="*50)
    print("🏆 FINAL TEST SCORES (QWK)")
    print("="*50)
        
    qwk_scores = []
    for i, trait in enumerate(TRAITS):
        trait_preds = all_preds[:, i]
        trait_targets = all_targets[:, i]
        qwk = compute_qwk(trait_preds, trait_targets)
        qwk_scores.append(qwk)
        clean_name = trait.replace("(ground_truth)", "")
        print(f"-> {clean_name:<25} : {qwk:.4f}")
        
    print("-" * 50)
    print(f"-> {'RATA-RATA QWK':<25} : {np.mean(qwk_scores):.4f}")
    print("="*50)

    # --- EKSPOR HASIL KE CSV ---
    print("\n[INFO] Memproses pembuatan file CSV...")
    pred_columns = [t.replace("(ground_truth)", "(prediction)") for t in TRAITS]
    df_preds = pd.DataFrame(all_preds, columns=pred_columns)

    meta_cols = [c for c in ['Question', 'Essay', 'graph', 'local_image_id'] if c in df_test.columns]
    df_meta = df_test[meta_cols].copy().reset_index(drop=True)
    
    df_final_csv = pd.concat([df_meta, df_preds], axis=1)
    
    output_filename = "clip_predictions_slidingwindow.csv"
    df_final_csv.to_csv(output_filename, index=False)
    
    print(f"✅ SUKSES! File hasil prediksi udah disimpan sebagai: '{output_filename}'")

if __name__ == "__main__":
    main()


c:\Users\gabyg\miniconda3\envs\skripsi\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


🔥 Using device: xpu
Loading Processor & Preparing Data...


The image processor of type `CLIPImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


📂 Loading predefined data split dari train_df.csv dan test_df.csv...
✅ Train set: 867 samples | Test set: 187 samples

Dataset siap: 867 Train | 187 Test

Initializing Model & Optimizer...


Loading weights: 100%|██████████| 398/398 [00:01<00:00, 388.65it/s, Materializing param=visual_projection.weight]                                
CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
text_model.embeddings.position_ids   | UNEXPECTED |  | 
vision_model.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


...

Epoch 9 Selesai | Avg Loss: 0.2418



Epoch [10/10] Train: 100%|██████████| 109/109 [02:02<00:00,  1.13s/it, loss=0.249]


Epoch 10 Selesai | Avg Loss: 0.2206

🔍 MULAI TESTING PADA DATA UNSEEN


Testing: 100%|██████████| 24/24 [00:26<00:00,  1.09s/it]



🏆 FINAL TEST SCORES (QWK)
-> organizational_structure  : 0.2683
-> coherence                 : 0.2740
-> essay_length              : 0.2884
-> grammatical_accuracy      : 0.3885
-> grammatical_diversity     : 0.3065
-> lexical_accuracy          : 0.2891
-> lexical_diversity         : 0.2936
-> punctuation_accuracy      : 0.3676
-> argument_clarity          : 0.1278
-> justifying_persuasiveness : 0.2327
--------------------------------------------------
-> RATA-RATA QWK             : 0.2837

[INFO] Memproses pembuatan file CSV...
✅ SUKSES! File hasil prediksi udah disimpan sebagai: 'clip_predictions_slidingwindow.csv'
